# 🏗️ Lezione 2: Il Full Stack Agentico

**Scuola Dati — dbt + Orchestrazione + Agenti AI**

> La prima lezione ci ha dato il **semantic layer** come contratto sui dati.  
> Questa lezione costruisce lo stack completo intorno ad esso.

## Obiettivi

1. Capire i **tre layer** dello stack agentico moderno
2. Vedere come **Dagster** orchestra dbt e triggera gli agenti
3. Implementare i tre pattern fondamentali: **tool calling**, **context injection**, **agent loop**
4. Costruire un agente di **monitoring con anomaly detection**


## 1. Architettura: Il Full Stack Agentico

```
  SORGENTI DATI
       │
       ▼
  ┌────────────────────────────────────────────────────────────┐
  │  LAYER 1 — CONTRATTO DEI DATI  (dbt)                      │
  │                                                            │
  │  raw seeds ──► staging views ──► fact tables              │
  │                                       │                   │
  │                          _semantic_layer.yml              │
  │                     (metriche, misure, dimensioni)        │
  └────────────────────────────────────────────────────────────┘
                                          │ asset pronti
                                          ▼
  ┌────────────────────────────────────────────────────────────┐
  │  LAYER 2 — ORCHESTRAZIONE  (Dagster / Airflow)            │
  │                                                            │
  │  Schedule ──► dbt run ──► dbt test ──► [SUCCESS/FAIL]    │
  │                                              │            │
  │                                         Sensor           │
  │                                     (rileva l'evento)    │
  └────────────────────────────────────────────────────────────┘
                                                 │ trigger
                                                 ▼
  ┌────────────────────────────────────────────────────────────┐
  │  LAYER 3 — INTELLIGENZA  (AI Agent)                       │
  │                                                            │
  │  @system_prompt: corpus dbt  ← context injection         │
  │  Tool 1: esegui_query()      ← SQL sul DB                │
  │  Tool 2: leggi_soglie_alert() ← config soglie            │
  │                                                            │
  │  Loop: controlla metrica ──► anomalia? ──► alert          │
  │         ↑                                      │          │
  │         └──────── prossima iterazione ◄──────┘          │
  └────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
                       OUTPUT STRUTTURATO
                   SegnalazioneMonitoraggio
             (severità, azione consigliata, SQL usato)
```


### Responsabilità di Ogni Layer

| Layer | Tecnologia | Responsabilità | Output |
|---|---|---|---|
| **Contratto** | dbt | Logica corretta, test, versioning | Tabelle testate + YAML metriche |
| **Orchestrazione** | Dagster/Airflow | Scheduling, retry, lineage, eventi | Run history, asset graph |
| **Intelligenza** | Pydantic AI + LLM | Ragionamento, decisioni, alert | Strutture Pydantic tipizzate |

> **Separazione delle responsabilità**: nessun layer fa il lavoro dell'altro.  
> dbt non orchestra. Dagster non ragiona. L'agente non trasforma dati.


## Installazione

### 1. Dipendenze base

```bash
uv sync
```

Installa: dbt-duckdb, pydantic-ai, anthropic, python-dotenv, sentence-transformers, numpy, jupyter e tutte le dipendenze transitive.

### 2. API Key Anthropic

Crea un file `.env` nella root del progetto (già in `.gitignore`):

```
ANTHROPIC_API_KEY=sk-ant-...
```

La cella di setup qui sotto lo carica automaticamente con `load_dotenv`.

### 3. Dagster (opzionale)

Richiesto solo per le celle Dagster (Layer 2):

```bash
uv sync --extra dagster
```

### 4. Verifica

```bash
# Verifica dbt e database (dalla cartella adventureworks/)
uv run dbt compile --profiles-dir ..

# Verifica agente (dalla root)
uv run python -c "from pydantic_ai import Agent; print('ok')"

# Verifica Dagster (opzionale)
uv run python -c "import dagster; print(dagster.__version__)"
```

In [1]:
import os
from pathlib import Path

import duckdb
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext

# ── Root del progetto ──────────────────────────────────────────────────
_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "adventureworks" / "dbt_project.yml").is_file() else _cwd.parent
if not (ROOT / "adventureworks" / "dbt_project.yml").is_file():
    raise FileNotFoundError(f"Progetto dbt non trovato sotto {_cwd}")

MODELS_DIR = ROOT / "adventureworks" / "models"
DUCKDB_PATH = str(ROOT / "adventureworks" / "data" / "adventureworks.duckdb")

print(f"✅ Root   : {ROOT}")
print(f"✅ DB     : {DUCKDB_PATH}")

# ── API key da .env ─────────────────────────────────────────────────────
load_dotenv(ROOT / ".env")
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    print(
        "\n❌ ANTHROPIC_API_KEY mancante — aggiungila a .env nella root del progetto:"
    )
    print("   ANTHROPIC_API_KEY=sk-ant-...")
else:
    print(f"\n✅ ANTHROPIC_API_KEY: {api_key[:12]}...{api_key[-4:]}")

MODEL = os.environ.get("PYDANTIC_AI_MODEL", "anthropic:claude-sonnet-4-6")
print(f"   Modello : {MODEL}")

✅ Root   : /Users/rmaganza/repos/scuoladati-semantic-modeling
✅ DB     : /Users/rmaganza/repos/scuoladati-semantic-modeling/adventureworks/data/adventureworks.duckdb

✅ ANTHROPIC_API_KEY: sk-ant-api03...PgAA
   Modello : anthropic:claude-sonnet-4-6


## 2. Layer 1 — dbt: Il Contratto

Il corpus che l'agente leggerà comprende tutti i file del progetto dbt.
Non è solo documentazione: è la **specifica formale** delle regole di business.


In [2]:
# ── Il corpus dbt visto dall'agente ──────────────────────────────────
print("Corpus del semantic layer:\n")
total_bytes = 0
for ext in ("*.yml", "*.sql"):
    for f in sorted(MODELS_DIR.rglob(ext)):
        if "target" not in str(f):
            rel = str(f.relative_to(ROOT))
            size = f.stat().st_size
            total_bytes += size
            print(f"  {rel:<60}  {size:>5} bytes")
print(f"\nKnowledge base dell'agente: {total_bytes:,} bytes")

Corpus del semantic layer:

  adventureworks/models/marts/_semantic_layer.yml                2311 bytes
  adventureworks/models/marts/_time_spine.yml                     298 bytes
  adventureworks/models/schema.yml                               4051 bytes
  adventureworks/models/marts/fct_customers.sql                   537 bytes
  adventureworks/models/marts/fct_orders.sql                      713 bytes
  adventureworks/models/marts/fct_products.sql                    696 bytes
  adventureworks/models/marts/time_spine_daily.sql                253 bytes
  adventureworks/models/staging/stg_categories.sql                157 bytes
  adventureworks/models/staging/stg_customers.sql                 139 bytes
  adventureworks/models/staging/stg_order_lines.sql               577 bytes
  adventureworks/models/staging/stg_orders.sql                    229 bytes
  adventureworks/models/staging/stg_products.sql                  143 bytes

Knowledge base dell'agente: 10,104 bytes


## 3. Layer 2 — Dagster: L'Orchestratore

Dagster è il collante tra il contratto dbt e l'intelligenza dell'agente.

### Concetti Chiave

| Concetto | Significato |
|---|---|
| **Asset** | Un artefatto prodotto dal pipeline (es. la tabella `fct_orders`) |
| **Job** | Un insieme di asset da materializzare insieme |
| **Schedule** | Una regola temporale per eseguire un job (`0 6 * * *`) |
| **Sensor** | Osservatore che triggera un job quando si verifica un evento |
| **Run** | Singola esecuzione di un job, con stato e log |

Con `dagster-dbt`, ogni modello dbt diventa automaticamente un **asset** Dagster:
- Dagster conosce la **lineage** (dipendenze tra modelli)
- Se `stg_order_lines` fallisce, Dagster non esegue `fct_orders`
- Il Sensor rileva ogni run completato e triggera l'agente

Il codice completo è in `lezione2/dagster_pipeline.py`.  
La cella seguente lo esegue direttamente: `dbt compile` genera il manifest,  
poi `dagster asset materialize` materializza i fact tables ed esce senza web server.

> **Nota**: richiede `uv sync --extra dagster` se Dagster non è ancora installato.

In [3]:
import subprocess

# ── Verifica installazione Dagster ────────────────────────────────────────
try:
    import dagster
    import dagster_dbt

    print(f"✅ dagster {dagster.__version__}  |  dagster-dbt {dagster_dbt.__version__}")
except ImportError:
    print("❌ Dagster non installato. Esegui nel terminale:")
    print("   uv sync --extra dagster")
    raise

# ── Genera il manifest dbt (richiesto da @dbt_assets al caricamento) ─────
# dbt compile deve girare da dentro adventureworks/ perché profiles.yml
# usa un path relativo (data/adventureworks.duckdb).
print("\n[1/2] dbt compile → manifest.json ...")
r = subprocess.run(
    ["dbt", "compile", "--profiles-dir", ".."],
    capture_output=True,
    text=True,
    cwd=str(ROOT / "adventureworks"),
)
print("      " + ("✅ ok" if r.returncode == 0 else f"❌ {r.stderr[-300:]}"))

# ── Materializza i tre fact table ─────────────────────────────────────────
# 'dagster asset materialize' è il comando moderno (sostituisce il deprecato
# 'dagster job execute'). Seleziona esattamente i tre asset del job.
print("\n[2/2] dagster asset materialize → fct_customers, fct_orders, fct_products ...")
r = subprocess.run(
    [
        "dagster",
        "asset",
        "materialize",
        "-f",
        str(ROOT / "lezione2" / "dagster_pipeline.py"),
        "--select",
        "fct_customers,fct_orders,fct_products",
    ],
    capture_output=True,
    text=True,
    cwd=str(ROOT),
    timeout=300,
)

output = (r.stdout + r.stderr).strip()
print("\n" + output[-2500:] if len(output) > 2500 else "\n" + output)
print(
    "\n✅ Job completato." if r.returncode == 0 else f"\n❌ Exit code: {r.returncode}"
)

✅ dagster 1.13.6  |  dagster-dbt 0.29.6

[1/2] dbt compile → manifest.json ...
      ✅ ok

[2/2] dagster asset materialize → fct_customers, fct_orders, fct_products ...

UTPUT - Yielded output "fct_customers" of type "Nothing". (Type check passed).
2026-05-28 17:46:00 +0200 - dagster - DEBUG - __ASSET_JOB - e1689416-36e9-495e-a9ae-b4e95fe0b824 - 2411 - adventureworks_assets - ASSET_MATERIALIZATION - Materialized value fct_customers.
2026-05-28 17:46:00 +0200 - dagster - DEBUG - __ASSET_JOB - e1689416-36e9-495e-a9ae-b4e95fe0b824 - 2411 - adventureworks_assets - STEP_OUTPUT - Yielded output "fct_products" of type "Nothing". (Type check passed).
2026-05-28 17:46:00 +0200 - dagster - DEBUG - __ASSET_JOB - e1689416-36e9-495e-a9ae-b4e95fe0b824 - 2411 - adventureworks_assets - ASSET_MATERIALIZATION - Materialized value fct_products.
2026-05-28 17:46:00 +0200 - dagster - DEBUG - __ASSET_JOB - e1689416-36e9-495e-a9ae-b4e95fe0b824 - 2411 - adventureworks_assets - STEP_OUTPUT - Yielded output "fc

### Cosa è Appena Successo

`dagster job execute` ha eseguito `dbt_refresh_job`, che materializza i tre fact table:

- **`fct_customers`** — un record per cliente, con `order_count` e `lifetime_net_revenue`
- **`fct_orders`** — un record per ordine spedito, con `net_revenue` calcolato dalle righe
- **`fct_products`** — un record per prodotto, con `units_sold` e `gross_revenue`

Il database `adventureworks.duckdb` è ora aggiornato con i dati più recenti.
Nella parte restante del notebook l'agente leggerà da questo database per calcolare le metriche.

In produzione questo step viene triggerato automaticamente dal `post_run_agent_sensor`
dopo ogni run dbt completato con successo — senza intervento manuale.


### Il Punto di Integrazione: Sensor → Agent

```
  [dbt run SUCCESS]
        │
        ▼
  Dagster Event Log
        │
        ▼
  post_run_agent_sensor
    (polling ogni 30s)
        │ RunRequest
        ▼
  Agent Monitoring Job [← questo notebook simula questo passaggio]
        │
        ▼
  [SegnalazioneMonitoraggio] ──► Slack / PagerDuty / Dashboard
```

In questo notebook **simuliamo** il trigger chiamando direttamente l'agente,
esattamente come farebbe il Dagster job.


## 4. Layer 3 — L'Agente: Tre Pattern

### Pattern 1 — Tool Calling su Dati Strutturati
L'agente chiama funzioni Python tipizzate per interrogare il database.
Il docstring di ogni tool è la documentazione che l'LLM legge per decidere quando usarlo.

### Pattern 2 — Context Injection
Il corpus dbt (YAML + SQL) viene iniettato **interamente** nel system prompt
tramite `@agent.system_prompt`. Il modello legge formule e definizioni
direttamente nel contesto, senza tool di ricerca.

> **Quando funziona**: corpus piccolo (< ~100K token).
> Il nostro corpus dbt occupa meno del 3% della context window di Claude.
> Per corpus grandi serve RAG con embeddings — vedi l'**Appendice** a fine notebook.

### Pattern 3 — Agent Loop con Anomaly Detection
L'agente gira in loop, controlla metriche, confronta con soglie,
produce alert strutturati. Il loop simula il job Dagster post-pipeline.


### Pydantic AI: i Mattoni Fondamentali

**`Agent`** è il punto di ingresso. Riceve un modello LLM, conosce i tool disponibili
e gestisce il ciclo autonomo: *ragiona → chiama tool → osserva il risultato → ragiona di nuovo → ...*
finché non ha abbastanza informazioni per produrre l'output richiesto.

**`output_type`** definisce la forma dell'output. Invece di testo libero, l'agente
produce un oggetto Pydantic validato. Il codice downstream può fare
`alert.severity == "critical"` invece di parsare una stringa — più robusto e tipizzato.

**`deps_type`** è la dependency injection. I tool hanno bisogno di risorse (database, corpus)
che non possono ricevere come argomenti normali — quelli li sceglie l'LLM.
La soluzione: si passa un oggetto `deps` a `.run(deps=...)`, e ogni tool lo riceve via `ctx.deps`.

**Tool** è una funzione Python decorata con `@agent.tool`. L'LLM non vede il codice:
vede solo nome, docstring e schema dei parametri, e decide autonomamente se e quando chiamarla.

```python
agent = Agent(
    "anthropic:claude-sonnet-4-6",
    deps_type=AgentStack,        # risorse iniettate a runtime
    output_type=MonitoringAlert, # Pydantic model dell'output finale
)

@agent.tool
def execute_query(ctx: RunContext[AgentStack], sql: str) -> str:
    """L'LLM legge questo docstring per decidere quando usare il tool."""
    ...
```


In [4]:
# ── Output strutturato: MonitoringAlert ──────────────────────────────────
from typing import Literal


class MonitoringAlert(BaseModel):
    """Alert strutturato con catena di derivazione dal modello semantico."""

    metric: str = Field(description="Nome della metrica (es. 'total_net_revenue')")
    current_value: str = Field(description="Valore corrente con unità")
    warning_threshold: str = Field(description="Soglia di warning con unità")
    severity: Literal["ok", "warning", "critical"] = Field(
        description="Stato: ok=nella norma, warning=attenzione, critical=urgente"
    )
    description: str = Field(description="Spiegazione in linguaggio naturale")
    recommended_action: str | None = Field(
        None, description="Azione raccomandata se severity != ok"
    )
    executed_sql: str = Field(description="Query SQL usata per il calcolo")
    model_source: str | None = Field(
        None, description="File dbt dove è definita la metrica"
    )


print("Struttura MonitoringAlert:")
for name, field in MonitoringAlert.model_fields.items():
    tipo = str(field.annotation).replace("typing.", "")
    print(f"  {name:<28}: {tipo}")

Struttura MonitoringAlert:
  metric                      : <class 'str'>
  current_value               : <class 'str'>
  warning_threshold           : <class 'str'>
  severity                    : Literal['ok', 'warning', 'critical']
  description                 : <class 'str'>
  recommended_action          : str | None
  executed_sql                : <class 'str'>
  model_source                : str | None


In [5]:
# ── Classe dipendenze ─────────────────────────────────────────────────────
class AgentStack:
    """Contesto iniettato nell'agente: database + corpus dbt."""

    def __init__(self, root: Path) -> None:
        self.root = root
        self.models_dir = root / "adventureworks" / "models"
        self.db_path = str(root / "adventureworks" / "data" / "adventureworks.duckdb")
        self._corpus: dict[str, str] | None = None

    @property
    def corpus(self) -> dict[str, str]:
        """Carica YAML/SQL del progetto come dizionario {path: contenuto}."""
        if self._corpus is None:
            self._corpus = {}
            for ext in ("*.yml", "*.sql"):
                for f in sorted(self.models_dir.rglob(ext)):
                    if "target" not in str(f) and "__pycache__" not in str(f):
                        key = str(f.relative_to(self.root))
                        self._corpus[key] = f.read_text()
        return self._corpus

    @property
    def corpus_text(self) -> str:
        """Corpus dbt concatenato, pronto per il system prompt."""
        return "\n\n".join(
            f"[{fname}]\n{content}" for fname, content in self.corpus.items()
        )


# ── Agente ────────────────────────────────────────────────────────────────
monitoring_agent = Agent(
    MODEL,
    deps_type=AgentStack,
    output_type=MonitoringAlert,
    system_prompt="""
    Sei un agente di monitoring dati. Dopo ogni run del pipeline dbt,
    controlli le metriche chiave e rilevi anomalie.

    PROCESSO PER OGNI METRICA:
    1. Trova la definizione della metrica nel corpus dbt (fornito nel contesto).
    2. Leggi le soglie di alert (read_alert_thresholds).
    3. Calcola il valore con una query SQL corretta (execute_query).
    4. Confronta con le soglie e determina la severity.
    5. Compila tutti i campi di MonitoringAlert.

    REGOLE:
    - executed_sql deve contenere la query effettivamente eseguita.
    - model_source indica il file dbt con la definizione della metrica.
    - recommended_action è obbligatoria se severity != 'ok'.
    - Rispondi sempre in italiano.
    """,
)


# ── Pattern 2: Context Injection ─────────────────────────────────────────
@monitoring_agent.system_prompt
def inject_corpus(ctx: RunContext[AgentStack]) -> str:
    """Inietta il corpus dbt completo nel system prompt a runtime."""
    return (
        f"CORPUS DBT (semantic layer, formule, definizioni):\n\n{ctx.deps.corpus_text}"
    )


# ── Pattern 1: Tool calling su dati strutturati ───────────────────────────
@monitoring_agent.tool
def execute_query(ctx: RunContext[AgentStack], sql: str) -> str:
    """
    Esegue una query SQL sul database DuckDB e restituisce il risultato.
    Scrivi query usando la logica trovata nel corpus dbt (formule, filtri).
    """
    try:
        con = duckdb.connect(ctx.deps.db_path, read_only=True)
        df = con.execute(sql).df()
        con.close()
        return df.to_string(index=False) if not df.empty else "Nessun risultato."
    except Exception as e:
        return f"Errore SQL: {e}"


# ── Configurazione soglie ─────────────────────────────────────────────────
@monitoring_agent.tool
def read_alert_thresholds(ctx: RunContext[AgentStack]) -> dict:
    """
    Restituisce le soglie di alert per le metriche monitorate.
    Confronta i valori calcolati con questi threshold per determinare la severity.
    """
    return {
        "total_net_revenue": {
            "warning": 12_000,
            "critical": 8_000,
            "unita": "EUR",
            "note": "Ricavo netto, solo ordini spediti (status=5)",
        },
        "order_count": {
            "warning": 5,
            "critical": 2,
            "unita": "ordini",
            "note": "Numero totale di ordini spediti (status=5)",
        },
        "avg_net_revenue_per_order": {
            "warning": 1_500,
            "critical": 1_000,
            "unita": "EUR",
            "note": "AOV netto = total_net_revenue / order_count",
        },
    }


print("✅ Agente monitor creato:")
print("   inject_corpus          (Pattern 2: corpus dbt nel system prompt)")
print("   execute_query          (Pattern 1: Tool Calling)")
print("   read_alert_thresholds  (configurazione soglie)")

✅ Agente monitor creato:
   inject_corpus          (Pattern 2: corpus dbt nel system prompt)
   execute_query          (Pattern 1: Tool Calling)
   read_alert_thresholds  (configurazione soglie)


## 5. Pattern 2 in Dettaglio: Context Injection

Il corpus dbt viene iniettato nel system prompt tramite il decorator `@agent.system_prompt`.
Pydantic AI lo chiama a runtime passando `ctx.deps`, così il corpus è sempre aggiornato.

```python
@monitoring_agent.system_prompt
def inject_corpus(ctx: RunContext[AgentStack]) -> str:
    return f"CORPUS DBT:\n\n{ctx.deps.corpus_text}"
```

Più `@system_prompt` sullo stesso agente si concatenano automaticamente —
il prompt fisso (istruzioni) e quello dinamico (corpus) vengono uniti prima di ogni chiamata.

La cella seguente mostra quanto spazio occupa il corpus nella context window.


In [6]:
# ── Demo: context injection ───────────────────────────────────────────────
stack = AgentStack(ROOT)

n_files = len(stack.corpus)
n_chars = len(stack.corpus_text)
n_tokens = n_chars // 4  # stima: ~4 caratteri per token

print(
    f"Corpus dbt : {n_files} file  |  {n_chars:,} caratteri  |  ~{n_tokens:,} token stimati"
)
print(f"Context window Claude Sonnet: 200,000 token")
print(f"Utilizzo contesto corpus    : {n_tokens / 200_000 * 100:.1f}%")
print("\n--- Anteprima corpus (prime 20 righe) ---\n")
for line in stack.corpus_text.splitlines()[:20]:
    print(line)

Corpus dbt : 12 file  |  10,690 caratteri  |  ~2,672 token stimati
Context window Claude Sonnet: 200,000 token
Utilizzo contesto corpus    : 1.3%

--- Anteprima corpus (prime 20 righe) ---

[adventureworks/models/marts/_semantic_layer.yml]
# MetricFlow / dbt Semantic Layer
# Dopo modifiche: dbt parse (o dbt run), poi mf validate-configs / mf query
#
# Nota: le misure richiedono una dimensione temporale di default (qui order_date).
# Metriche su fct_customers / fct_products si possono aggiungere con un time spine
# dedicato o incrociando entity customer da questo modello.

semantic_models:
  - name: orders_semantic
    description: |
      Fact ordini (una riga per ordine shipped). Entity customer per join nel grafo semantico.
    model: ref('fct_orders')
    defaults:
      agg_time_dimension: order_date
    entities:
      - name: order
        type: primary
        expr: order_id
      - name: customer


## 6. Pattern 3: Il Loop di Monitoring

Le tre metriche che il loop controllerà — tutte definite in `_semantic_layer.yml`:

| # | Metrica | Definita in | Soglia warning | Esito atteso |
|---|---|---|---|---|
| 1 | `total_net_revenue` | `_semantic_layer.yml` | €12.000 | ⚠️ warning (~€9.740) |
| 2 | `order_count` | `_semantic_layer.yml` | 5 ordini | ✅ ok (6 ordini) |
| 3 | `avg_net_revenue_per_order` | `_semantic_layer.yml` | €1.500 | ✅ ok (~€1.623) |

> L'ordine 9 è in stato 3 (in lavorazione): i suoi €4.200 non rientrano nel calcolo
> perché `fct_orders` filtra solo gli ordini spediti (`status = 5`).
> Questo simula un giorno in cui un ordine importante è ancora in sospeso.

In [7]:
# ── Funzione del loop ────────────────────────────────────────────────────
async def monitoring_loop(
    metrics: list[str],
    stack: AgentStack,
) -> list[MonitoringAlert]:
    """
    Simula il Dagster job che gira dopo ogni run dbt.
    In produzione: triggerato dal post_run_agent_sensor.
    """
    alerts = []
    print("\U0001f504 MONITORING LOOP AVVIATO")
    print("   (simulazione del Dagster agent job)\n")

    sep = "─" * 62
    for i, metric in enumerate(metrics, 1):
        print(sep)
        print(f"  Iterazione {i}/{len(metrics)}  →  {metric}")
        print(sep)

        result = await monitoring_agent.run(
            f"Controlla la metrica '{metric}'. "
            "Processo: 1) trova definizione nel corpus dbt, "
            "2) leggi le soglie (read_alert_thresholds), "
            "3) calcola il valore con SQL corretto (execute_query), "
            "4) determina severity e compila MonitoringAlert.",
            deps=stack,
        )
        alert = result.output
        alerts.append(alert)

        icons = {"ok": "✅", "warning": "⚠️", "critical": "\U0001f6a8"}
        icon = icons[alert.severity]
        print(f"\n{icon}  {alert.metric.upper()}  [{alert.severity.upper()}]")
        print(
            f"   Valore: {alert.current_value}  |  Soglia warning: {alert.warning_threshold}"
        )
        print(f"   {alert.description}")
        if alert.recommended_action:
            print(f"   ▶ Azione: {alert.recommended_action}")
        if alert.model_source:
            print(f"   \U0001f4c4 Fonte: {alert.model_source}")
        print()

    return alerts

### Due Loop Sovrapposti

Il sistema ha due loop distinti — è importante non confonderli:

**Loop esterno** (`monitoring_loop`) — codice che controlliamo noi:
itera sulle metriche e chiama `monitoring_agent.run(...)` una volta per ciascuna.

**Loop interno** — gestito automaticamente da Pydantic AI:
per ogni chiamata a `.run(...)`, l'LLM decide quante volte invocare i tool
e in quale ordine, finché non ha abbastanza dati per compilare `MonitoringAlert`.

```
Loop esterno (nostro for):
  metric 1 → monitoring_agent.run()
               └─ loop interno (Pydantic AI):
                    LLM chiama read_alert_thresholds()  → osserva
                    LLM chiama execute_query(sql1)      → osserva
                    LLM chiama execute_query(sql2)      → osserva  ← può riprovare
                    LLM produce MonitoringAlert         → fine
  metric 2 → monitoring_agent.run()  ...
  metric 3 → monitoring_agent.run()  ...
```

Il loop interno è trasparente: il codice esterno riceve solo il risultato finale.
L'LLM può chiamare `execute_query` più volte (es. se la prima query è sbagliata)
senza che il loop esterno ne sia a conoscenza.


In [8]:
# ── Esecuzione del loop ──────────────────────────────────────────
# Tutte e tre definite in adventureworks/models/marts/_semantic_layer.yml
METRICS_TO_MONITOR = [
    "total_net_revenue",
    "order_count",
    "avg_net_revenue_per_order",
]

alerts = await monitoring_loop(METRICS_TO_MONITOR, stack)

🔄 MONITORING LOOP AVVIATO
   (simulazione del Dagster agent job)

──────────────────────────────────────────────────────────────
  Iterazione 1/3  →  total_net_revenue
──────────────────────────────────────────────────────────────

⚠️  TOTAL_NET_REVENUE  [WARNING]
   Valore: 9.740,00 EUR  |  Soglia warning: 12.000 EUR (critical: 8.000 EUR)
   La revenue netta totale sugli ordini spediti (status=5) è pari a 9.740,00 EUR. Il valore è al di sotto della soglia di warning (12.000 EUR) ma ancora sopra quella critica (8.000 EUR). La metrica è definita nel semantic layer come somma della misura `net_revenue` su `fct_orders`, calcolata dalle righe ordine con la formula `line_total * (1 - discount_pct)`. Il segnale di warning indica un calo significativo della revenue netta che richiede attenzione immediata.
   ▶ Azione: 1. Verificare se ci sono ordini spediti recenti non ancora processati o bloccati in stati intermedi (es. status=2 processing). 2. Analizzare la distribuzione degli sconti (disco

In [9]:
# ── Riepilogo: dashboard testuale ───────────────────────────────────────
print("═" * 62)
print("  RIEPILOGO MONITORING RUN")
print("═" * 62)

n_ok = sum(1 for a in alerts if a.severity == "ok")
n_warning = sum(1 for a in alerts if a.severity == "warning")
n_critical = sum(1 for a in alerts if a.severity == "critical")

print(f"\n  ✅ OK       : {n_ok}")
print(f"  ⚠️  Warning  : {n_warning}")
print(f"  \U0001f6a8 Critical : {n_critical}")

if n_warning + n_critical > 0:
    print("\n  Alert da inviare (Slack / PagerDuty):")
    for alert in alerts:
        if alert.severity != "ok":
            print(f"\n  [{alert.severity.upper()}] {alert.metric}")
            print(f"   Valore: {alert.current_value}")
            print(f"   {alert.description}")
            if alert.recommended_action:
                print(f"   Azione: {alert.recommended_action}")
else:
    print("\n  Tutte le metriche nella norma. Nessun alert.")

print("\n" + "═" * 62)

══════════════════════════════════════════════════════════════
  RIEPILOGO MONITORING RUN
══════════════════════════════════════════════════════════════

  ✅ OK       : 2
  ⚠️  Warning  : 1
  🚨 Critical : 0

  Alert da inviare (Slack / PagerDuty):

  [WARNING] total_net_revenue
   Valore: 9.740,00 EUR
   La revenue netta totale sugli ordini spediti (status=5) è pari a 9.740,00 EUR. Il valore è al di sotto della soglia di warning (12.000 EUR) ma ancora sopra quella critica (8.000 EUR). La metrica è definita nel semantic layer come somma della misura `net_revenue` su `fct_orders`, calcolata dalle righe ordine con la formula `line_total * (1 - discount_pct)`. Il segnale di warning indica un calo significativo della revenue netta che richiede attenzione immediata.
   Azione: 1. Verificare se ci sono ordini spediti recenti non ancora processati o bloccati in stati intermedi (es. status=2 processing). 2. Analizzare la distribuzione degli sconti (discount_pct) nelle order_lines per rilevare s

### Output Strutturato: esplorare gli oggetti `MonitoringAlert`

Il print sopra mostra testo formattato a mano. Ma `alerts` è una lista di oggetti Pydantic — ogni campo è accessibile direttamente in Python, senza parsare stringhe.

In [ ]:
import json

# ── Accesso diretto ai campi Pydantic ─────────────────────────────────────
print("Accesso diretto ai campi:\n")
for alert in alerts:
    print(f"  alert.metric    = {alert.metric!r}")
    print(f"  alert.severity  = {alert.severity!r}")
    print(f"  alert.current_value     = {alert.current_value!r}")
    print(f"  alert.warning_threshold = {alert.warning_threshold!r}")
    print()

# ── Filtrare per severity in una riga ────────────────────────────────────
warnings = [a for a in alerts if a.severity != "ok"]
print(f"Alert non-ok: {len(warnings)} su {len(alerts)}\n")

# ── Il campo executed_sql: la query che l'LLM ha scritto autonomamente ───
print("SQL generato dall'agente per ogni metrica:\n")
for alert in alerts:
    print(f"── {alert.metric} ──")
    print(alert.executed_sql)
    print()

# ── Serializzazione a dict / JSON ────────────────────────────────────────
print("Serializzazione a dict (primo alert):\n")
print(json.dumps(alerts[0].model_dump(), indent=2, ensure_ascii=False))

## 7. Come Appare in Produzione

```
  Cloud Storage (S3/GCS)
        │ dbt source
        ▼
  dbt Core / dbt Cloud  (CI + scheduled run)
        │ artefatti (manifest.json, run_results.json)
        ▼
  Dagster  (post_run_agent_sensor)
        │ RunRequest
        ▼
  Agent Service  (FastAPI + Pydantic AI)
        │ SegnalazioneMonitoraggio JSON
        ├──► Slack webhook
        ├──► PagerDuty API
        └──► Data catalog (Atlan / DataHub)
```

| Aspetto | Notebook | Produzione |
|---|---|---|
| **Trigger** | Chiamata diretta | Dagster sensor |
| **Database** | DuckDB locale | Snowflake / BigQuery |
| **Corpus RAG** | File su disco | dbt Cloud API / metadata store |
| **Alert** | `print()` | Slack / PagerDuty |
| **Scalabilità** | 1 agente sequenziale | N agenti in parallelo (`asyncio.gather`) |


## 8. I Tre Pattern Riassunti

### Pattern 1 — Tool Calling su Dati Strutturati
L'agente chiama SQL tools tipizzati. Il LLM scrive la query,
il tool la esegue e restituisce dati freschi dal database.

### Pattern 2 — Context Injection
Il corpus dbt viene iniettato interamente nel system prompt via `@agent.system_prompt`.
Il modello legge formule e definizioni direttamente nel contesto, senza tool di ricerca.
Funziona perché il corpus è piccolo. Per corpus grandi: vedi Appendice RAG.

### Pattern 3 — Agent Loop
Il loop simula la risposta a eventi del pipeline.
Ogni iterazione è indipendente → scalabile a N agenti in parallelo.

> **I tre pattern si combinano**:
> il loop (3) chiama l'agente → il corpus nel contesto (2) fornisce le formule
> → il tool calling (1) calcola i valori → produce alert strutturato.


## Appendice: RAG con Embeddings

### Context Injection vs RAG

| | Context Injection | RAG con Embeddings |
|---|---|---|
| **Retrieval** | Nessuno — il modello vede tutto | Similarità coseno sugli embeddings |
| **Quando usarlo** | Corpus piccolo (< ~100K token) | Corpus troppo grande per il contesto |
| **Complessità** | Minima | Richiede embedding model + indice |
| **Qualità** | Il modello vede tutto | Dipende dalla qualità del retrieval |
| **Dipendenze extra** | Nessuna | `sentence-transformers`, `numpy` |

---

### I Passaggi Matematici

#### 1. Embedding: testo → vettore

Un **embedding model** (qui `all-MiniLM-L6-v2`) trasforma un testo in un vettore di numeri reali:

```
"net_revenue = line_total * (1 - discount_pct)"
        │
        ▼  modello neurale
        │
[0.12, -0.34, 0.87, ..., -0.22]   ← vettore in ℝ³⁸⁴
```

Testi con **significato simile** producono vettori che puntano nella stessa direzione. Testi non correlati puntano in direzioni diverse.

#### 2. Cosa fa il codice: dot product su vettori normalizzati

Guarda `rag_search`:

```python
q_emb = model.encode([query], normalize_embeddings=True)   # ‖q‖ = 1
scores = (embeddings @ q_emb.T).squeeze()                  # dot product puro
```

Il codice esegue un **dot product** (prodotto interno), non la formula completa della cosine similarity. È corretto perché `normalize_embeddings=True` impone che ogni vettore abbia lunghezza 1.

La cosine similarity è definita come:

```
         A · B
cos(θ) = ─────────
         ‖A‖ · ‖B‖
```

Quando ‖A‖ = ‖B‖ = 1 (vettori normalizzati), il denominatore è 1 e la formula si riduce al solo numeratore:

```
cos(θ) = A · B     ← identico a embeddings @ q_emb.T
```

Quindi **dot product su vettori normalizzati = cosine similarity**. Il risultato misura l'angolo θ tra i due vettori:

- `1.0` → stesso angolo, testi quasi identici nel significato
- `0.0` → angolo retto, testi non correlati
- valori negativi → rari per testi in linguaggio naturale

#### 3. Batch retrieval: una moltiplicazione di matrici

Con `N` chunk nell'indice, si calcolano tutti gli score in una sola operazione:

```
Matrice embeddings    Query embedding    Score per chunk
    (N × 384)      ×     (384 × 1)    =     (N × 1)

 [──── e₁ ────]                         [e₁ · q]
 [──── e₂ ────]  ×  [q₁, q₂, ..., q₃₈₄]ᵀ  =  [e₂ · q]
 [    ...     ]                         [  ...  ]
 [──── eₙ ────]                         [eₙ · q]
```

`numpy` esegue questa operazione come una singola chiamata BLAS — più efficiente di un loop Python su N dot product separati.

#### 4. Pipeline completa con numeri concreti

```
INDEXING (una volta):

  12 file dbt  →  ~30 chunk (finestre 20 righe, overlap 5)
               →  encode(normalize=True)
               →  matrice (30 × 384) float32  ≈ 46 KB

RETRIEVAL (per ogni query):

  "formula revenue con sconti"
    → encode(normalize=True)  → vettore (384,)
    → embeddings @ q.T        → scores (30,)   ← dot product = cosine similarity
    → argsort descending       → top-3 indici
    → chunk 7  [0.91]  fct_orders.sql
      chunk 2  [0.87]  _semantic_layer.yml
      chunk 9  [0.74]  stg_order_lines.sql
    → concatenati nel prompt dell'LLM
```

#### Nota sui modelli

| Modello | Dim | Peso | Note |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | ~90 MB | veloce, buono per corpus tecnici in inglese |
| `all-mpnet-base-v2` | 768 | ~420 MB | più preciso per testi lunghi |
| `text-embedding-3-small` (API) | 1536 | — | ottimo, richiede chiave API |

Per corpus dbt in italiano la qualità scende perché MiniLM è addestrato prevalentemente su inglese. In produzione si usa un modello multilingue o l'embedding API del proprio provider LLM.

---

### Come Funziona nel Codice

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# ── Parametri ─────────────────────────────────────────────────────────────
EMBED_MODEL = "all-MiniLM-L6-v2"  # ~90 MB, scaricato una volta in ~/.cache
CHUNK_LINES = 20
CHUNK_OVERLAP = 5


def build_rag_index(corpus: dict[str, str]):
    """Chunking + embedding del corpus. Ritorna (model, chunks, embeddings)."""
    chunks = []
    for filename, content in corpus.items():
        lines = [l for l in content.splitlines() if l.strip()]
        if len(lines) <= CHUNK_LINES:
            chunks.append((filename, content))
        else:
            step = CHUNK_LINES - CHUNK_OVERLAP
            for i in range(0, len(lines), step):
                window = "\n".join(lines[i : i + CHUNK_LINES])
                chunks.append((filename, window))

    model = SentenceTransformer(EMBED_MODEL)
    texts = [f"[{fname}]\n{chunk}" for fname, chunk in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)
    return model, chunks, embeddings


def rag_search(query: str, model, chunks, embeddings, top_k: int = 3):
    """Ricerca semantica per similarità coseno."""
    q_emb = model.encode([query], normalize_embeddings=True)
    scores = (embeddings @ q_emb.T).squeeze()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i][0], chunks[i][1], float(scores[i])) for i in top_idx]


# ── Costruzione indice ────────────────────────────────────────────────────
print("Costruzione indice RAG (prima esecuzione: scarica ~90 MB)...")
rag_model, rag_chunks, rag_embeddings = build_rag_index(stack.corpus)
print(
    f"  {len(stack.corpus)} file  →  {len(rag_chunks)} chunk  →  {rag_embeddings.shape[1]}d embeddings\n"
)

# ── Ricerca semantica ─────────────────────────────────────────────────────
for query in ["net revenue formula discount", "customers without orders"]:
    print(f"Query: {repr(query)}")
    for fname, chunk, score in rag_search(query, rag_model, rag_chunks, rag_embeddings):
        print(f"  [{score:.2f}]  {fname}")
        for line in chunk.splitlines()[:4]:
            print(f"    {line}")
    print()

Costruzione indice RAG (prima esecuzione: scarica ~90 MB)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

## 9. Esercizi

### Base

1. **Nuova metrica**: Aggiungi `tasso_cancellazioni` al loop  
   (% ordini con `status = 6`). Soglie: warning=10%, critical=25%.

2. **Campo Slack**: Aggiungi `slack_message: str` a `MonitoringAlert`.  
   L'agente lo compila con un testo pronto per l'invio.

3. **RAG con scoring**: Migliora il tool RAG dando più peso alle righe  
   che contengono *tutti* i termini della query, non solo uno.

### Intermedio

4. **Dagster reale**: Installa con `uv sync --extra dagster`.  
   Lancia `dagster dev -f lezione2/dagster_pipeline.py` e verifica il lineage graph.

5. **Multi-agente parallelo**: Esegui i check in parallelo con `asyncio.gather()`.  
   Confronta il tempo di esecuzione con il loop sequenziale.

6. **Soglie dinamiche**: Invece di soglie fisse, fai calcolare la media  
   degli ultimi periodi e usa quella come soglia adattiva.

### Avanzato

7. **Self-healing**: Aggiungi `dbt_tests_to_run: list[str]` all'output.  
   Quando l'agente rileva un'anomalia, suggerisce quale test dbt rieseguire.

8. **API REST**: Esponi il monitoring loop come `POST /monitoring/run`  
   con FastAPI. Il body contiene le metriche, la risposta è la lista di alert.

9. **Loop completo**: Collega il Sensor Dagster (esercizio 4) all'API REST  
   (esercizio 8). Dagster triggera il sensor → chiama l'endpoint → salva  
   gli alert in una tabella `monitoring_alerts` nel database.